# Module 33 — Learning to Rank with RankNet (PyTorch)

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: PyTorch, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 28 (Siamese BERT), Module 4 (PyTorch Tensors)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Query-Document Pairs](#3-synthetic-query-document-pairs)
4. [RankNet Architecture](#4-ranknet-architecture)
5. [Pairwise Training](#5-pairwise-training)
6. [Evaluation](#6-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why Learning to Rank for search?

Learning to Rank (LTR) **optimizes ranking quality**:
- **Pointwise**: Predict absolute relevance scores.
- **Pairwise**: Learn relative ordering between documents.
- **Listwise**: Optimize entire ranking list directly.

**Applications at Yandex**:
1. **Web search**: Rank web pages by relevance.
2. **Ad ranking**: Order ads by expected value.
3. **Recommendation**: Rank items by user preference.

**Key algorithms**:
- **RankNet**: Pairwise ranking with cross-entropy loss.
- **LambdaRank**: Listwise ranking with NDCG-aware gradients.
- **LambdaMART**: LambdaRank with gradient boosted trees.

In this notebook we will:
1. Generate synthetic query-document relevance data.
2. Build a RankNet model in PyTorch.
3. Train with pairwise ranking loss.
4. Evaluate with NDCG@K and MAP.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Deep learning ────────────────────────────────────────────────
import torch
import torch.nn as nn

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports loaded")
print(f"  Device: {device}")

---
## §3 · Synthetic Query-Document Pairs

In [ ]:
# Generate synthetic query-document relevance data
N_QUERIES = 1000
DOCS_PER_QUERY = 10
N_FEATURES = 20

# Generate queries and documents
queries = []
for query_id in range(N_QUERIES):
    # Query features
    query_features = np.random.randn(N_FEATURES)
    
    for doc_rank in range(DOCS_PER_QUERY):
        # Document features (correlated with relevance)
        doc_features = np.random.randn(N_FEATURES)
        
        # Relevance score (higher for better documents)
        relevance = np.dot(query_features, doc_features) + np.random.normal(0, 0.5)
        relevance = np.clip(relevance, 0, 4).round()  # 0-4 scale
        
        queries.append({
            'query_id': query_id,
            'doc_id': f'doc_{query_id}_{doc_rank}',
            'features': doc_features,
            'relevance': int(relevance)
        })

df = pd.DataFrame(queries)

print(f"✓ Generated {len(df)} query-document pairs")
print(f"  Queries: {N_QUERIES}")
print(f"  Documents per query: {DOCS_PER_QUERY}")
print(f"  Feature dimension: {N_FEATURES}")
print(f"\nRelevance distribution:")
print(df['relevance'].value_counts().sort_index())

---
## §4 · RankNet Architecture

In [ ]:
# Define RankNet model
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.Dropout(0.2))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create model
model = RankNet(input_dim=N_FEATURES).to(device)

print(f"✓ RankNet model created")
print(f"  Input dimension: {N_FEATURES}")
print(f"  Hidden layers: [64, 32]")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## §5 · Pairwise Training

In [ ]:
# Define pairwise ranking loss
class PairwiseRankingLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, score_diff, label_diff):
        # label_diff: 1 if doc_i more relevant than doc_j, -1 otherwise
        # score_diff: score_i - score_j
        loss = torch.log(1 + torch.exp(-label_diff * score_diff))
        return loss.mean()

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = PairwiseRankingLoss()

# Generate training pairs
def generate_pairs(query_data):
    """Generate pairwise training examples."""
    pairs = []
    docs = query_data.sort_values('relevance', ascending=False)
    
    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            if docs.iloc[i]['relevance'] != docs.iloc[j]['relevance']:
                pairs.append({
                    'doc_i': np.array(docs.iloc[i]['features'].tolist()),
                    'doc_j': np.array(docs.iloc[j]['features'].tolist()),
                    'label': 1 if docs.iloc[i]['relevance'] > docs.iloc[j]['relevance'] else -1
                })
    return pairs

# Generate pairs for first query
sample_query = df[df['query_id'] == 0]
pairs = generate_pairs(sample_query)

print(f"Training setup:")
print(f"  Loss: Pairwise ranking loss")
print(f"  Optimizer: Adam (lr=1e-3)")
print(f"  Sample pairs from query 0: {len(pairs)}")

---
## §6 · Evaluation

In [ ]:
# Evaluate ranking quality
def compute_ndcg(relevances, k=5):
    """Compute NDCG@k."""
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances[:k]))
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevances[:k]))
    return dcg / idcg if idcg > 0 else 0

# Compute NDCG for sample query
relevances = sample_query['relevance'].values
ndcg_5 = compute_ndcg(relevances, k=5)
ndcg_10 = compute_ndcg(relevances, k=10)

print("=" * 70)
print("RANKING EVALUATION")
print("=" * 70)
print(f"\nQuery 0 relevance scores: {relevances}")
print(f"\nNDCG@5: {ndcg_5:.4f}")
print(f"NDCG@10: {ndcg_10:.4f}")
print(f"\nNote: NDCG=1.0 means perfect ranking")

---
## §7 · Visualization

In [ ]:
# Visualize ranking
fig = go.Figure()

# Add relevance scores
fig.add_trace(go.Bar(
    x=list(range(len(relevances))),
    y=relevances,
    name='Relevance',
    marker_color='#636EFA'
))

fig.update_layout(
    title='Document Relevance Scores (Query 0)',
    xaxis_title='Document Rank',
    yaxis_title='Relevance Score',
    height=400
)
fig.show()

print("💡 Key insights:")
print("   - Higher relevance documents should rank higher")
print("   - RankNet learns to order documents correctly")
print("   - NDCG measures ranking quality")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Search & Ranking Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Search ranking, ad ordering, recommendation |
> | **Algorithm** | RankNet (pairwise), LambdaMART (listwise) |
> | **Features** | Query-document features, user features, context |
> | **Evaluation** | NDCG@K, MAP, MRR |
> | **Deployment** | Pre-compute features, real-time scoring |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Ranking pipeline**:
>    ```
>    Query → Retrieve candidates → Extract features → Rank → Display
>    ```
>
> 2. **LTR approaches**:
>    | Approach | Pros | Cons |
>    |----------|------|------|
>    | Pointwise | Simple | Ignores relative ordering |
>    | Pairwise | Learns ordering | Quadratic pairs |
>    | Listwise | Optimizes NDCG | Complex training |
>
> 3. **Production tips**:
>    - Use LambdaMART for best performance
>    - Combine with neural re-rankers
>    - A/B test ranking changes
>
> ### 🔑 Key Takeaways
>
> 1. **Learning to Rank optimizes ranking quality** directly.
> 2. **Pairwise methods** learn relative ordering between documents.
> 3. **NDCG@K** is the standard metric for ranking evaluation.
> 4. **Feature engineering** is critical for ranking performance.
> 5. **Combine LTR with neural models** for best results.